# L3 — Refund Agent

## What You'll Learn

- How to create a Lambda with multiple tools and expose them through the AgentCore Gateway
- How to query a Bedrock Knowledge Base from a Lambda tool
- How to deploy a Strands agent as an A2A server on AgentCore Runtime
- How to register an agent in the AgentCore Registry so the Harness can discover it

## Overview

The Refund Agent handles all refund-related requests from AnyCompany customers. It exposes three tools: `refund_policy` (retrieves return policy from the Bedrock Knowledge Base), `check_eligible` (evaluates whether an order qualifies for a refund), and `process_refund` (generates a refund confirmation). This notebook follows the same pattern as the Order Agent — Lambda → Gateway Target → Runtime → Registry.

## Architecture

```
  Harness (Orchestrator)
       │
       │  discovers via AgentCore Registry
       ▼
  ┌─────────────────────────────────────────────────────┐
  │ AgentCore Runtime container (sidecar image)          │
  │                                                     │
  │  supervisord                                        │
  │   ├─ Datadog trace-agent  ──► Datadog APM intake   │
  │   └─ ddtrace-run refund_agent_a2a (Strands, A2A)   │
  │         │  APM + LLM Obs spans → localhost:8126     │
  └─────────┼───────────────────────────────────────────┘
             │  calls tools via AgentCore Gateway (MCP)
             ▼
  Lambda: anycompany_refund_tools
       ├── refund_policy  ──► Bedrock Knowledge Base (return policy + product images)
       ├── check_eligible ──► eligibility logic (order status + damage evidence)
       └── process_refund ──► generates refund ID and ETA
```

## Steps

1. Create the Lambda execution role and deploy the refund tools Lambda
2. Create the Gateway Target (MCP tool schemas)
3. Deploy the Refund Agent to AgentCore Runtime
4. Test the agent via the Runtime endpoint
5. Register the agent in the AgentCore Registry
6. Save resource IDs to SSM

## License Details

Refer to LICENSE file in the main folder for license details.

## Prerequisites

- `1_pre-requisites.ipynb` completed — Registry, Gateway, and Cognito IDs must be in SSM
- `1_pluggable_inference_layer.ipynb` completed — `model_id` must be in SSM
- `1_s3_knowledge_base_setup.ipynb` completed — `kb_id` must be in SSM
- `boto3` and `strands-agents-tools` (installed in the first code cell)

Install required packages.

In [1]:
%pip install --quiet --upgrade boto3 strands-agents-tools==0.2.0 bedrock-agentcore==1.14.0 bedrock-agentcore-starter-toolkit==0.3.6

Note: you may need to restart the kernel to use updated packages.


Import libraries, set the region, and load shared configuration from SSM.

In [2]:
import boto3
import json
import time
import os
import io
import zipfile
import uuid
from datetime import datetime, timedelta

REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1") # Change to your preferred region
SSM_PREFIX = "/anycompany/agentcore"

ssm = boto3.client("ssm", region_name=REGION)
iam = boto3.client("iam")
sts = boto3.client("sts", region_name=REGION)
lambda_client = boto3.client("lambda", region_name=REGION)
agentcore_control = boto3.client("bedrock-agentcore-control", region_name=REGION)
agent_registry = boto3.client("agent-registry-control", region_name=REGION)

def get_ssm(name, default=None):
    try:
        return ssm.get_parameter(Name=name, WithDecryption=True)["Parameter"]["Value"]
    except ssm.exceptions.ParameterNotFound:
        if default is not None:
            return default
        raise

ACCOUNT_ID = sts.get_caller_identity()["Account"]
REGISTRY_ID = get_ssm(f"{SSM_PREFIX}/registry_id")
GATEWAY_ID = get_ssm(f"{SSM_PREFIX}/gateway_id")
GATEWAY_ARN = get_ssm(f"{SSM_PREFIX}/gateway_arn")
GATEWAY_URL = get_ssm(f"{SSM_PREFIX}/gateway_url")
MODEL_ID = get_ssm(f"{SSM_PREFIX}/model_id")
BEDROCK_KB_ID = get_ssm(f"{SSM_PREFIX}/kb_id", default="")

print(f"Account:     {ACCOUNT_ID}")
print(f"Region:      {REGION}")
print(f"Registry:    {REGISTRY_ID}")
print(f"Gateway:     {GATEWAY_ID}")
print(f"Gateway URL: {GATEWAY_URL}")
print(f"Model:       {MODEL_ID}")
print(f"KB ID:       {BEDROCK_KB_ID or '(not set)'}")

Account:     359894365202
Region:      us-west-2
Registry:    XCtvGlP9E1u5ZoYr
Gateway:     anycompany-agent-gateway-m1jnn7lvju
Gateway URL: https://anycompany-agent-gateway-m1jnn7lvju.gateway.bedrock-agentcore.us-west-2.amazonaws.com/mcp
Model:       bedrock/us.anthropic.claude-sonnet-4-6
KB ID:       BAQBCKQYIC


## Step 1: Create Lambda Functions for Refund Tools

Create the IAM execution role for the Lambda with permissions for CloudWatch Logs, DynamoDB, and Bedrock Knowledge Base retrieval.

In [3]:
LAMBDA_ROLE_NAME = "AnyCompanyRefundToolsLambdaRole"

lambda_trust = json.dumps({
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "lambda.amazonaws.com"},
        "Action": "sts:AssumeRole"
    }]
})

lambda_perms = json.dumps({
    "Version": "2012-10-17",
    "Statement": [
        {"Effect": "Allow", "Action": ["logs:CreateLogGroup"], "Resource": f"arn:aws:logs:*:{ACCOUNT_ID}:log-group:/aws/lambda/anycompany_refund_tools"},
        {"Effect": "Allow", "Action": ["logs:CreateLogStream", "logs:PutLogEvents"], "Resource": f"arn:aws:logs:*:{ACCOUNT_ID}:log-group:/aws/lambda/anycompany_refund_tools:*"},
        {"Effect": "Allow", "Action": ["bedrock:Retrieve", "bedrock:RetrieveAndGenerate"], "Resource": f"arn:aws:bedrock:*:{ACCOUNT_ID}:knowledge-base/*"},
        {"Effect": "Allow", "Action": ["dynamodb:GetItem", "dynamodb:Query", "dynamodb:UpdateItem"], "Resource": [f"arn:aws:dynamodb:*:{ACCOUNT_ID}:table/CustomerOrders", f"arn:aws:dynamodb:*:{ACCOUNT_ID}:table/CustomerOrders/index/*"]}
    ]
})

try:
    r = iam.create_role(RoleName=LAMBDA_ROLE_NAME, AssumeRolePolicyDocument=lambda_trust)
    LAMBDA_ROLE_ARN = r["Role"]["Arn"]
    iam.put_role_policy(RoleName=LAMBDA_ROLE_NAME, PolicyName="RefundToolsPerms", PolicyDocument=lambda_perms)
    print(f"Created: {LAMBDA_ROLE_ARN}")
    time.sleep(10)
except iam.exceptions.EntityAlreadyExistsException:
    LAMBDA_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{LAMBDA_ROLE_NAME}"
    print(f"Exists: {LAMBDA_ROLE_ARN}")

Created: arn:aws:iam::359894365202:role/AnyCompanyRefundToolsLambdaRole


Package the Lambda handler code as a zip and deploy `anycompany_refund_tools` (Python 3.12). Grants the Gateway permission to invoke it.

In [4]:
LAMBDA_CODE = '''import json, os, time, boto3, logging
from datetime import datetime, timedelta

logger = logging.getLogger()
logger.setLevel(logging.INFO)

REGION = os.environ.get("AGENT_REGION", "us-east-1")
KB_ID = os.environ.get("BEDROCK_KB_ID", "")
kb_client = boto3.client("bedrock-agent-runtime", region_name=REGION) if KB_ID else None


def query_kb(query, max_results=3):
    if not KB_ID or not kb_client:
        return "Knowledge Base not configured."
    resp = kb_client.retrieve(
        knowledgeBaseId=KB_ID,
        retrievalQuery={"text": query},
        retrievalConfiguration={"vectorSearchConfiguration": {"numberOfResults": max_results}},
    )
    passages = [r.get("content", {}).get("text", "") for r in resp.get("retrievalResults", []) if r.get("content", {}).get("text")]
    return "\\n---\\n".join(passages) if passages else "No relevant policy found."


def handle_refund_policy(params):
    return {"status": "success", "policy_info": query_kb(params.get("question", ""))}


def handle_check_eligible(params):
    order_id = params.get("order_id", "")
    order_status = params.get("order_status", "")
    reason = params.get("reason", "")
    damage_level = params.get("damage_level", "")
    eligible_statuses = {"delivered", "shipped", "return_requested"}
    if order_status.lower() not in eligible_statuses:
        return {"eligible": "no", "order_id": order_id, "message": f"Status '{order_status}' not eligible."}
    if reason.lower() in ["damaged on arrival", "defective product"] and not damage_level:
        return {"eligible": "pending_evidence", "order_id": order_id, "message": "Photo evidence required."}
    return {"eligible": "yes", "order_id": order_id, "message": f"Order {order_id} is eligible for refund."}


def handle_process_refund(params):
    order_id = params.get("order_id", "")
    amount = float(params.get("refund_amount", 0))
    reason = params.get("reason", "")
    prefix = order_id.replace("ORD-", "")
    refund_id = f"REF-{prefix}-{int(time.time())}"
    eta = (datetime.utcnow() + timedelta(days=5)).strftime("%Y-%m-%d")
    return {"refund_approved": True, "refund_id": refund_id, "order_id": order_id, "amount": f"${amount:.2f}", "eta": eta, "reason": reason}


def lambda_handler(event, context):
    """Handle tool calls from Gateway.
    
    Gateway passes tool arguments directly as the event payload.
    We detect which tool based on the parameters present.
    """
    logger.info(f"Tool invoked - request_id: {context.aws_request_id}")
    
    if "question" in event:
        result = handle_refund_policy(event)
    elif "refund_amount" in event:
        result = handle_process_refund(event)
    elif "order_id" in event:
        result = handle_check_eligible(event)
    else:
        result = {"error": f"Cannot determine tool from event keys: {list(event.keys())}"}
    
    logger.info(f"Tool completed - status: {result.get('status', 'unknown')}")
    return result
'''

# Package as zip
buf = io.BytesIO()
with zipfile.ZipFile(buf, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.writestr('lambda_function.py', LAMBDA_CODE)
zip_bytes = buf.getvalue()

LAMBDA_NAME = "anycompany_refund_tools"

try:
    resp = lambda_client.create_function(
        FunctionName=LAMBDA_NAME,
        Runtime="python3.12",
        Role=LAMBDA_ROLE_ARN,
        Handler="lambda_function.lambda_handler",
        Code={"ZipFile": zip_bytes},
        Timeout=60,
        MemorySize=256,
        Environment={"Variables": {"BEDROCK_KB_ID": BEDROCK_KB_ID, "AGENT_REGION": REGION}},
    )
    LAMBDA_ARN = resp["FunctionArn"]
    print(f"Created: {LAMBDA_ARN}")
except lambda_client.exceptions.ResourceConflictException:
    LAMBDA_ARN = lambda_client.get_function(FunctionName=LAMBDA_NAME)["Configuration"]["FunctionArn"]
    lambda_client.update_function_code(FunctionName=LAMBDA_NAME, ZipFile=zip_bytes)
    # Wait for code update to settle, then refresh env vars so a recreated KB ID is picked up.
    lambda_client.get_waiter('function_updated_v2').wait(FunctionName=LAMBDA_NAME)
    lambda_client.update_function_configuration(
        FunctionName=LAMBDA_NAME,
        Environment={"Variables": {"BEDROCK_KB_ID": BEDROCK_KB_ID, "AGENT_REGION": REGION}},
    )
    print(f"Updated: {LAMBDA_ARN}")

# Allow Gateway to invoke
try:
    lambda_client.add_permission(
        FunctionName=LAMBDA_NAME,
        StatementId="AllowGatewayInvoke",
        Action="lambda:InvokeFunction",
        Principal="bedrock-agentcore.amazonaws.com",
    )
    print("Added Gateway invoke permission.")
except lambda_client.exceptions.ResourceConflictException:
    print("Permission already exists.")

Created: arn:aws:lambda:us-west-2:359894365202:function:anycompany_refund_tools
Added Gateway invoke permission.


## Step 2: Create Gateway Target

Register the three refund tools as an MCP target on the Gateway. Waits for Lambda to be Active, then waits for the target to reach READY.

In [5]:
TOOL_SCHEMAS = [
    {
        "name": "refund_policy",
        "description": "Retrieve refund policy information from the knowledge base. Use when the customer asks about return windows, eligible categories, or policy rules.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "question": {"type": "string", "description": "The policy question to look up"}
            },
            "required": ["question"]
        }
    },
    {
        "name": "check_eligible",
        "description": "Check whether an order is eligible for a refund based on status, reason, and damage evidence.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string", "description": "Order ID (e.g. ORD-12345)"},
                "order_status": {"type": "string", "description": "Current status (delivered, shipped, etc.)"},
                "item": {"type": "string", "description": "Product name"},
                "reason": {"type": "string", "description": "Reason for refund request"},
                "damage_level": {"type": "string", "description": "Damage severity (minor/moderate/severe)"}
            },
            "required": ["order_id", "order_status", "item", "reason"]
        }
    },
    {
        "name": "process_refund",
        "description": "Process an approved refund. Only call after eligibility is confirmed.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string", "description": "Order ID"},
                "refund_amount": {"type": "number", "description": "Dollar amount to refund"},
                "reason": {"type": "string", "description": "Reason for the refund"}
            },
            "required": ["order_id", "refund_amount", "reason"]
        }
    }
]

TARGET_NAME = "refund-tools"

# Wait for Lambda to be fully active before creating the Gateway Target
print("Waiting for Lambda to be active...")
for _ in range(20):
    fn = lambda_client.get_function_configuration(FunctionName=LAMBDA_NAME)
    state = fn.get("State", "")
    last_update = fn.get("LastUpdateStatus", "Successful")
    if state == "Active" and last_update in ("Successful", None):
        print(f"Lambda is Active and ready.")
        break
    print(f"  State: {state}, LastUpdateStatus: {last_update} — waiting 5s...")
    time.sleep(5)
else:
    raise TimeoutError("Lambda did not become active within 100s.")

try:
    target_resp = agentcore_control.create_gateway_target(
        gatewayIdentifier=GATEWAY_ID,
        name=TARGET_NAME,
        description="Refund tools (policy, eligibility, processing) backed by Lambda",
        targetConfiguration={
            "mcp": {
                "lambda": {
                    "lambdaArn": LAMBDA_ARN,
                    "toolSchema": {"inlinePayload": TOOL_SCHEMAS}
                }
            }
        },
        credentialProviderConfigurations=[
            {"credentialProviderType": "GATEWAY_IAM_ROLE"}
        ],
    )
    TARGET_ID = target_resp["targetId"]
    print(f"Created Gateway Target: {TARGET_ID}")
except agentcore_control.exceptions.ConflictException:
    targets = agentcore_control.list_gateway_targets(gatewayIdentifier=GATEWAY_ID)
    TARGET_ID = None
    for t in targets.get("targets", targets.get("items", [])):
        if t.get("name") == TARGET_NAME:
            TARGET_ID = t["targetId"]
            print(f"Already exists: {TARGET_ID}")
            break
    if not TARGET_ID:
        # Try getting by name directly
        t = agentcore_control.get_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=TARGET_NAME)
        TARGET_ID = t["targetId"]
        print(f"Found by name: {TARGET_ID}")

# Wait for ready
print("Waiting for target...")
for _ in range(20):
    t = agentcore_control.get_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=TARGET_ID)
    status = t["status"]
    if status == "READY":
        print(f"Gateway Target READY")
        break
    print(f"  {status}...")
    time.sleep(5)

print(f"\nTools available via Gateway: refund_policy, check_eligible, process_refund")

Waiting for Lambda to be active...
  State: Pending, LastUpdateStatus: Successful — waiting 5s...
Lambda is Active and ready.
Created Gateway Target: JRYJMVRVOO
Waiting for target...
  CREATING...
Gateway Target READY

Tools available via Gateway: refund_policy, check_eligible, process_refund


## Step 3: Deploy to AgentCore Runtime

Create the Runtime IAM role, configure the agent with `refund_agent_a2a.py` as the entrypoint, and launch it as an A2A server.

In [6]:
# Remove cached config to avoid cross-notebook conflicts
import os
if os.path.exists(".bedrock_agentcore.yaml"):
    os.remove(".bedrock_agentcore.yaml")
    print("Cleared cached .bedrock_agentcore.yaml")
if os.path.exists("Dockerfile"):
    os.remove("Dockerfile")
    print("Removed stale Dockerfile")

# Stage the Datadog trace-agent sidecar Dockerfile
import shutil
shutil.copyfile("agents/sidecar/Dockerfile", "Dockerfile")
print("Staged sidecar Dockerfile (Datadog trace-agent)")

from bedrock_agentcore_starter_toolkit import Runtime

# IAM role for AgentCore Runtime
RUNTIME_ROLE_NAME = "AgentCoreRefundAgentA2ARole"

runtime_trust = json.dumps({
    "Version": "2012-10-17",
    "Statement": [{"Effect": "Allow", "Principal": {"Service": "bedrock-agentcore.amazonaws.com"}, "Action": "sts:AssumeRole"}]
})

try:
    r = iam.create_role(RoleName=RUNTIME_ROLE_NAME, AssumeRolePolicyDocument=runtime_trust, Description="Runtime role for Refund A2A Agent")
    RUNTIME_ROLE_ARN = r["Role"]["Arn"]
    print(f"Created: {RUNTIME_ROLE_ARN}")
except iam.exceptions.EntityAlreadyExistsException:
    RUNTIME_ROLE_ARN = iam.get_role(RoleName=RUNTIME_ROLE_NAME)["Role"]["Arn"]
    print(f"Exists: {RUNTIME_ROLE_ARN}")

# Least-privilege runtime execution role policy.
# Based on AWS-documented runtime permissions:
# https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-permissions.html
# X-Ray tracing requires Resource:* for PutTraceSegments/PutTelemetryRecords per AWS documentation
# CloudWatch metrics require Resource:* but namespace condition limits scope to bedrock-agentcore
iam.put_role_policy(
    RoleName=RUNTIME_ROLE_NAME,
    PolicyName="RefundAgentRuntimeAccess",
    PolicyDocument=json.dumps({
        "Version": "2012-10-17",
        "Statement": [
            {"Sid": "BedrockInference", "Effect": "Allow", "Action": ["bedrock:InvokeModel", "bedrock:InvokeModelWithResponseStream"], "Resource": ["arn:aws:bedrock:*::foundation-model/*", "arn:aws:bedrock:*:*:inference-profile/*"]},
            {"Sid": "ECRPublicAuth", "Effect": "Allow", "Action": ["ecr-public:GetAuthorizationToken", "sts:GetServiceBearerToken"], "Resource": "*"},
            {"Sid": "ECRAuth", "Effect": "Allow", "Action": ["ecr:GetAuthorizationToken"], "Resource": "*"},
            {"Sid": "ECRAgentImage", "Effect": "Allow", "Action": ["ecr:BatchGetImage", "ecr:GetDownloadUrlForLayer", "ecr:BatchCheckLayerAvailability"], "Resource": "arn:aws:ecr:*:*:repository/bedrock-agentcore-*"},
            {"Sid": "XRayTracing", "Effect": "Allow", "Action": ["xray:PutTraceSegments", "xray:PutTelemetryRecords", "xray:GetSamplingRules", "xray:GetSamplingTargets"], "Resource": "*"},
            {"Sid": "CloudWatchMetrics", "Effect": "Allow", "Action": "cloudwatch:PutMetricData", "Resource": "*", "Condition": {"StringEquals": {"cloudwatch:namespace": "bedrock-agentcore"}}},
            {"Sid": "CloudWatchLogsGroup", "Effect": "Allow", "Action": ["logs:CreateLogGroup", "logs:CreateLogStream", "logs:PutLogEvents", "logs:DescribeLogStreams"], "Resource": f"arn:aws:logs:*:{ACCOUNT_ID}:log-group:/aws/bedrock-agentcore/runtimes/*"},
            {"Sid": "CloudWatchLogsDescribeGroups", "Effect": "Allow", "Action": ["logs:DescribeLogGroups"], "Resource": f"arn:aws:logs:*:{ACCOUNT_ID}:log-group:*"},
            {"Sid": "AgentCoreWorkloadIdentity", "Effect": "Allow", "Action": ["bedrock-agentcore:GetWorkloadAccessToken", "bedrock-agentcore:GetWorkloadAccessTokenForJWT"], "Resource": [f"arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT_ID}:workload-identity-directory/default", f"arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT_ID}:workload-identity-directory/default/workload-identity/*"]},
            {"Sid": "SSMConfig", "Effect": "Allow", "Action": ["ssm:GetParameter"], "Resource": f"arn:aws:ssm:*:*:parameter{SSM_PREFIX}/*"},
        ]
    })
)
print(f"Runtime role ready: {RUNTIME_ROLE_ARN}")
time.sleep(10)

# Configure & Launch
agentcore_rt = Runtime()

configure_response = agentcore_rt.configure(
    entrypoint="agents/refund_agent_a2a.py",
    execution_role=RUNTIME_ROLE_ARN,
    auto_create_ecr=True,
    requirements_file="agents/requirements_refund_a2a.txt",
    region=REGION,
    agent_name="refund_agent_a2a",
    protocol="A2A",
    disable_otel=True,  # ADOT off - Datadog trace-agent sidecar owns telemetry
)
print(f"Configured: {configure_response}")

Cleared cached .bedrock_agentcore.yaml
Staged sidecar Dockerfile (Datadog trace-agent)
Created: arn:aws:iam::359894365202:role/AgentCoreRefundAgentA2ARole
Runtime role ready: arn:aws:iam::359894365202:role/AgentCoreRefundAgentA2ARole


Entrypoint parsed: file=/home/sagemaker-user/pluggable-agentic-ai-framework/l3-orchestration/agents/refund_agent_a2a.py, bedrock_agentcore_name=refund_agent_a2a
Memory disabled - agent will be stateless
Configuring BedrockAgentCore agent: refund_agent_a2a
Memory disabled
Network mode: PUBLIC


⚠️ Platform mismatch: Current system is 'linux/amd64' but Bedrock AgentCore requires 'linux/arm64', so local builds
won't work.
Please use default launch command which will do a remote cross-platform build using code build.For deployment other
options and workarounds, see: 
https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/getting-started-custom.html

📄 Using existing Dockerfile: /home/sagemaker-user/pluggable-agentic-ai-framework/l3-orchestration/Dockerfile

Setting 'refund_agent_a2a' as default agent
Bedrock AgentCore configured: /home/sagemaker-user/pluggable-agentic-ai-framework/l3-orchestration/.bedrock_agentcore.yaml


Configured: config_path=PosixPath('/home/sagemaker-user/pluggable-agentic-ai-framework/l3-orchestration/.bedrock_agentcore.yaml') dockerfile_path=PosixPath('/home/sagemaker-user/pluggable-agentic-ai-framework/l3-orchestration/Dockerfile') dockerignore_path=None runtime='Docker' runtime_type=None region='us-west-2' account_id='359894365202' execution_role='arn:aws:iam::359894365202:role/AgentCoreRefundAgentA2ARole' ecr_repository=None auto_create_ecr=True s3_path=None auto_create_s3=False memory_id=None network_mode='PUBLIC' network_subnets=None network_security_groups=None network_vpc_id=None


Launch the agent and poll until deployment reaches ACTIVE or READY.

In [7]:
# Deploy the agent with Datadog trace-agent sidecar.
# The sidecar ships APM spans (incl. AI Guard) to Datadog. ADOT is disabled.
#
# LLMOBS_ENABLED=true: self-contained (needs only DD_API_KEY, safe at deploy).
# AI_GUARD_ENABLED=false: requires L4 infrastructure (DynamoDB risk table,
#   DD_APP_KEY with ai_guard_evaluate scope) - activated later via env update.
DD_API_KEY = ssm.get_parameter(Name=f"{SSM_PREFIX}/dd_api_key", WithDecryption=True)["Parameter"]["Value"]
DD_SITE = ssm.get_parameter(Name=f"{SSM_PREFIX}/dd_site")["Parameter"]["Value"]

try:
    launch_result = agentcore_rt.launch(
        auto_update_on_conflict=True,
        env_vars={
            "AGENT_MODULE": "agents.refund_agent_a2a",
            "DD_API_KEY": DD_API_KEY,
            "DD_SITE": DD_SITE,
            "DD_SERVICE": "anycompany-refund-agent",
            "DD_ENV": "demo",
            "LLMOBS_ENABLED": "true",
            "DD_LLMOBS_ML_APP": "anycompany-customer-support",
            "AI_GUARD_ENABLED": "false",
        },
    )
    REFUND_AGENT_ARN = launch_result.agent_arn
    print(f"Launched: {REFUND_AGENT_ARN}")

    # Wait for deployment
    print("Waiting for deployment...", end="")
    for _ in range(60):
        status_response = agentcore_rt.status()
        deploy_status = status_response.endpoint["status"]
        if deploy_status in ["ACTIVE", "READY", "FAILED"]:
            break
        print(".", end="", flush=True)
        time.sleep(10)
    print(f"\nDeployment status: {deploy_status}")
    if deploy_status == "FAILED":
        print("Deployment failed. Check AgentCore console.")

finally:
    # Always clean up staged Dockerfile so it never hijacks a later deploy
    if os.path.exists("Dockerfile"):
        os.remove("Dockerfile")
        print("Removed staged sidecar Dockerfile.")


🚀 Launching Bedrock AgentCore (cloud mode - RECOMMENDED)...
   • Deploy Python code directly to runtime
   • No Docker required (DEFAULT behavior)
   • Production-ready deployment

💡 Deployment options:
   • runtime.launch()                → Cloud (current)
   • runtime.launch(local=True)      → Local development
Memory disabled - skipping memory creation
Starting CodeBuild ARM64 deployment for agent 'refund_agent_a2a' to account 359894365202 (us-west-2)
Generated image tag: 20261007-214140-810
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: refund_agent_a2a


Repository doesn't exist, creating new ECR repository: bedrock-agentcore-refund_agent_a2a


ECR repository available: 359894365202.dkr.ecr.us-west-2.amazonaws.com/bedrock-agentcore-refund_agent_a2a
Using execution role from config: arn:aws:iam::359894365202:role/AgentCoreRefundAgentA2ARole
Preparing CodeBuild project and uploading source...
Getting or creating CodeBuild execution role for agent: refund_agent_a2a
Role name: AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-cc1dfe14af
CodeBuild role doesn't exist, creating new role: AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-cc1dfe14af
Creating IAM role: AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-cc1dfe14af
✓ Role created: arn:aws:iam::359894365202:role/AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-cc1dfe14af
Attaching inline policy: CodeBuildExecutionPolicy to role: AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-cc1dfe14af
✓ Policy attached: CodeBuildExecutionPolicy
Waiting for IAM role propagation...
CodeBuild execution role creation complete: arn:aws:iam::359894365202:role/AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-cc1dfe14af


Launched: arn:aws:bedrock-agentcore:us-west-2:359894365202:runtime/refund_agent_a2a-uzrohQ9gru
Waiting for deployment...

Retrieved Bedrock AgentCore status for: refund_agent_a2a



Deployment status: READY
Removed staged sidecar Dockerfile.


## Step 4: Test the Agent

Invoke the deployed agent via the Runtime endpoint using a Cognito test token.

Authenticate as `gold_customer`, send an A2A JSON-RPC refund eligibility request, and print the response.

In [8]:
import requests
from urllib.parse import quote
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest

# Resolve REFUND_AGENT_ARN if not already set
if 'REFUND_AGENT_ARN' not in dir() or REFUND_AGENT_ARN is None:
    agentcore_runtime_client = boto3.client("bedrock-agentcore", region_name=REGION)
    runtimes = agentcore_runtime_client.list_agent_runtimes()
    REFUND_AGENT_ARN = None
    for rt in runtimes.get("agentRuntimes", runtimes.get("runtimes", [])):
        if "refund_agent_a2a" in rt.get("agentRuntimeArn", rt.get("arn", "")):
            REFUND_AGENT_ARN = rt.get("agentRuntimeArn", rt.get("arn"))
            break
    if not REFUND_AGENT_ARN:
        raise RuntimeError("Could not find refund_agent_a2a runtime. Please run Step 4b first.")
    print(f"Resolved agent ARN: {REFUND_AGENT_ARN}")

# Get a Cognito token for the test user (simulates orchestrator passing user token)
cognito_client = boto3.client("cognito-idp", region_name=REGION)
# Retrieve the workshop test password from SSM (stored as SecureString).
# get_ssm() passes WithDecryption=True so we get the plaintext back.
USER_PASSWORD = get_ssm(f"{SSM_PREFIX}/user_password")
auth_resp = cognito_client.initiate_auth(
    ClientId=get_ssm(f"{SSM_PREFIX}/cognito_client_id"),
# PRODUCTION REQUIREMENT: Replace USER_PASSWORD_AUTH with federated identity (SAML/OIDC).
# Shared passwords do not meet enterprise security requirements.
# See: https://docs.aws.amazon.com/cognito/latest/developerguide/cognito-user-pools-identity-federation.html
    AuthFlow="USER_PASSWORD_AUTH",
    # ⚠️ WORKSHOP ONLY — disposable Cognito test user; password fetched from SSM.
    # In production, use federated identity (SAML/OIDC) instead of shared passwords.
    AuthParameters={"USERNAME": "gold_customer", "PASSWORD": USER_PASSWORD},
)
test_token = auth_resp["AuthenticationResult"]["IdToken"]
print(f"Got Cognito token for gold_customer (test user)")

# Build the invocation URL
escaped_arn = quote(REFUND_AGENT_ARN, safe="")
invoke_url = f"https://bedrock-agentcore.{REGION}.amazonaws.com/runtimes/{escaped_arn}/invocations"

# Prepare A2A JSON-RPC request (message/send)
session_id = str(uuid.uuid4())
a2a_payload = json.dumps({
    "jsonrpc": "2.0",
    "method": "message/send",
    "id": str(uuid.uuid4()),
    "params": {
        "message": {
            "role": "user",
            "parts": [{"kind": "text", "text": "What is the refund policy for electronics?"}],
            "messageId": str(uuid.uuid4())
        }
    }
})

# Sign request with SigV4 (Runtime inbound) + pass test user token
session = boto3.Session(region_name=REGION)
creds = session.get_credentials().get_frozen_credentials()
headers = {
    "Content-Type": "application/json",
    "X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": session_id,
    "Authorization": f"Bearer {test_token}",
}
aws_req = AWSRequest(method="POST", url=invoke_url, data=a2a_payload, headers=headers)
SigV4Auth(creds, "bedrock-agentcore", REGION).add_auth(aws_req)

# Invoke the agent
print(f"Invoking Refund Agent via Runtime endpoint...")
print(f"  Query: What is the refund policy for electronics?")
print()

response = requests.post(invoke_url, data=a2a_payload, headers=dict(aws_req.headers), timeout=120)
print(f"Response status: {response.status_code}")

if response.status_code == 200:
    resp_text = response.text
    try:
        resp_json = json.loads(resp_text)
        if "error" in resp_json:
            print(f"\n❌ Error: {json.dumps(resp_json['error'], indent=2)}")
        elif "result" in resp_json:
            result = resp_json["result"]
            artifacts = result.get("artifacts", [])
            for artifact in artifacts:
                for part in artifact.get("parts", []):
                    if part.get("kind") == "text":
                        print(part["text"][:2000])
            print("\n✅ Agent invocation via Runtime endpoint successful!")
        else:
            print(json.dumps(resp_json, indent=2)[:2000])
    except json.JSONDecodeError:
        lines = [l for l in resp_text.strip().split("\n") if l.strip()]
        for line in lines:
            try:
                event = json.loads(line)
                if event.get("result", {}).get("status", {}).get("state") == "completed":
                    for artifact in event.get("result", {}).get("artifacts", []):
                        for part in artifact.get("parts", []):
                            if part.get("kind") == "text":
                                print(part["text"][:2000])
            except json.JSONDecodeError:
                pass
        print("\n✅ Agent invocation via Runtime endpoint successful!")
else:
    print(f"\n❌ Error: {response.status_code}")
    print(response.text[:1000])

Got Cognito token for gold_customer (test user)
Invoking Refund Agent via Runtime endpoint...
  Query: What is the refund policy for electronics?

Response status: 200
Here's a summary of the **Refund Policy for Electronics**:

---

### 📦 Return Conditions
- Items must be **unused and in original packaging**
- All **parts, accessories, and manuals** must be included
- No physical damage, stains, odors, or signs of use
- **Electronics must be wiped of all personal data**
- Serial numbers must match original purchase records

---

### 🕐 Return Windows (by Membership Tier)
| Tier | Return Window |
|------|--------------|
| Gold | 30 days from delivery |
| Silver | 20 days from delivery |
| Bronze | 15 days from delivery |

---

### 💰 Refund Percentages (by Membership Tier)
| Tier | Refund Amount |
|------|--------------|
| Gold | 100% of purchase price |
| Silver | 75% of purchase price |
| Bronze | Determined by product condition assessment |

---

### ⚠️ Special Cases
- **Defective item

## Step 5: Register in the AgentCore Registry

Fetch the A2A agent card from the deployed Runtime endpoint.

In [9]:
from urllib.parse import quote
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest
import requests

# Fetch the Agent Card from the deployed endpoint
print(REFUND_AGENT_ARN)
escaped_arn = quote(REFUND_AGENT_ARN, safe="")
card_url = f"https://bedrock-agentcore.{REGION}.amazonaws.com/runtimes/{escaped_arn}/invocations/.well-known/agent-card.json"

session = boto3.Session(region_name=REGION)
creds = session.get_credentials().get_frozen_credentials()
aws_req = AWSRequest(method="GET", url=card_url, headers={"X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": str(uuid.uuid4())})
SigV4Auth(creds, "bedrock-agentcore", REGION).add_auth(aws_req)

card_resp = requests.get(card_url, headers=dict(aws_req.headers))
card_resp.raise_for_status()
agent_card = card_resp.json()
print(f"Fetched Agent Card: {agent_card.get('name', 'unknown')}")
print(json.dumps(agent_card, indent=2)[:500])

arn:aws:bedrock-agentcore:us-west-2:359894365202:runtime/refund_agent_a2a-uzrohQ9gru
Fetched Agent Card: Refund Agent
{
  "capabilities": {
    "streaming": true
  },
  "defaultInputModes": [
    "text"
  ],
  "defaultOutputModes": [
    "text"
  ],
  "description": "Handles refund policy queries, eligibility evaluation, and refund processing.",
  "name": "Refund Agent",
  "preferredTransport": "JSONRPC",
  "protocolVersion": "0.3.0",
  "skills": [
    {
      "description": "Retrieve refund policy information from the knowledge base.",
      "id": "refund_policy",
      "name": "refund_policy",
      "tags": [


Create the registry record with the inline agent card and submit it for approval (auto-approved).

In [10]:
# Register in the Registry with inline agent card
RECORD_NAME = "refund_agent_a2a"

try:
    reg_resp = agent_registry.create_registry_record(
        registryId=REGISTRY_ID,
        name=RECORD_NAME,
        displayName="Refund Agent (A2A)",
        description="Refund Agent — handles policy queries, eligibility checks, and refund processing.",
        recordType="AGENT",
        recordVersion="1.0",
        descriptors={"a2aAgentCard": {"data": json.dumps(agent_card)}},
    )
    RECORD_ID = reg_resp["recordArn"].rsplit("/", 1)[-1]
    print(f"Created registry record: {RECORD_ID}")
except agent_registry.exceptions.ConflictException:
    print(f"Record '{RECORD_NAME}' already exists.")
    existing = agent_registry.list_registry_records(registryId=REGISTRY_ID)
    for rec in existing.get("registryRecords", []):
        if rec.get("name") == RECORD_NAME:
            RECORD_ID = rec["recordId"]
            print(f"  Found: {RECORD_ID}")
            break

# Wait for record to be ready
for _ in range(20):
    rec = agent_registry.get_registry_record(registryId=REGISTRY_ID, recordId=RECORD_ID)
    if rec["status"] not in ["CREATING", "UPDATING"]:
        break
    time.sleep(2)

# Submit for approval (auto-approved). Idempotent: skip if already APPROVED on re-runs.
if rec["status"] == "APPROVED":
    print(f"Record already APPROVED — no submission needed. Status: {rec['status']}")
else:
    agent_registry.submit_registry_record_for_approval(registryId=REGISTRY_ID, recordId=RECORD_ID)
    print(f"Record submitted for approval. Status: {rec['status']}")

Created registry record: yHX1b17F0uc2
Record submitted for approval. Status: DRAFT


## Step 6: Save Resource IDs to SSM

Publish the Lambda ARN and Gateway Target ID to SSM for use by L4, L5, and L6.

In [11]:
params = {
    f"{SSM_PREFIX}/refund_lambda_arn": LAMBDA_ARN,
    f"{SSM_PREFIX}/refund_gateway_target_id": TARGET_ID,
    f"{SSM_PREFIX}/refund_runtime_role_arn": RUNTIME_ROLE_ARN,  # read by L4/L5 (6_aiguard + sidecar) to redeploy this runtime
}
for name, value in params.items():
    ssm.put_parameter(Name=name, Value=value, Type="SecureString", Overwrite=True)
    print(f"  {name} = {value}")
print("\nDone.")

  /anycompany/agentcore/refund_lambda_arn = arn:aws:lambda:us-west-2:359894365202:function:anycompany_refund_tools
  /anycompany/agentcore/refund_gateway_target_id = JRYJMVRVOO
  /anycompany/agentcore/refund_runtime_role_arn = arn:aws:iam::359894365202:role/AgentCoreRefundAgentA2ARole

Done.


## Summary

| Resource | Name | Purpose |
|----------|------|---------|
| Lambda | `anycompany_refund_tools` | Handles `refund_policy`, `check_eligible`, and `process_refund` |
| Gateway Target | `refund-tools` | Exposes the three tools as MCP tools via the Gateway |
| AgentCore Runtime | `refund_agent_a2a` | Strands A2A agent deployed on Datadog trace-agent sidecar image |
| Datadog Telemetry | `anycompany-refund-agent` | APM traces + LLM Observability via in-container trace-agent |
| Registry Record | `refund_agent_a2a` | Makes the agent discoverable by the Harness |


---
## Cleanup

Run the following cell to delete all resources created in this notebook and avoid ongoing charges.

**Resources deleted:**
- AgentCore: runtime (sidecar image), registry record, gateway target
- Compute: Lambda function, ECR repository, CodeBuild project
- IAM: Lambda execution role and Runtime execution role
- Configuration: SSM parameters

Note: Datadog credentials (stored by `1_setup_resources.ipynb`) are NOT deleted here.


Uncomment and run to delete all resources created in this notebook.

In [12]:
## === CLEANUP: Delete all resources created in this notebook ===
## ⚠️ Uncomment and run this cell only after you are done with ALL notebooks.
## NOTE: Run this BEFORE the L3 #1 (1_setup_resources.ipynb) cleanup, so the
##       registry record drains before the registry itself is deleted.

# import boto3, os, time
#
# REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
# SSM_PREFIX = "/anycompany/agentcore"
#
# LAMBDA_NAME       = "anycompany_refund_tools"
# LAMBDA_ROLE_NAME  = "AnyCompanyRefundToolsLambdaRole"
# RUNTIME_ROLE_NAME = "AgentCoreRefundAgentA2ARole"
# TARGET_NAME       = "refund-tools"
# RUNTIME_NAME      = "refund_agent_a2a"
# RECORD_NAME       = "refund_agent_a2a"
# ECR_REPO_NAME     = "bedrock-agentcore-refund_agent_a2a"
# CODEBUILD_PROJECT = "bedrock-agentcore-refund_agent_a2a-builder"
#
# iam               = boto3.client("iam")
# lambda_client     = boto3.client("lambda", region_name=REGION)
# ssm               = boto3.client("ssm", region_name=REGION)
# agentcore_control = boto3.client("bedrock-agentcore-control", region_name=REGION)
# agent_registry    = boto3.client("agent-registry-control", region_name=REGION)
# logs_client       = boto3.client("logs", region_name=REGION)
# ecr               = boto3.client("ecr", region_name=REGION)
# codebuild         = boto3.client("codebuild", region_name=REGION)
#
# def _ssm_get(key):
#     try:
#         return ssm.get_parameter(Name=f"{SSM_PREFIX}/{key}")["Parameter"]["Value"]
#     except ssm.exceptions.ParameterNotFound:
#         return None
#
# GATEWAY_ID  = _ssm_get("gateway_id")
# REGISTRY_ID = _ssm_get("registry_id")
#
# # ── Phase 1: Delete Registry Record (must precede L3#1 registry delete) ─────
# if REGISTRY_ID:
#     try:
#         records = agent_registry.list_registry_records(registryId=REGISTRY_ID).get("registryRecords", [])
#         for rec in records:
#             if rec.get("name") == RECORD_NAME:
#                 try:
#                     agent_registry.delete_registry_record(registryId=REGISTRY_ID, recordId=rec["recordId"])
#                     print(f"Deleted registry record: {RECORD_NAME}")
#                 except agentcore_control.exceptions.ResourceNotFoundException:
#                     print(f"Registry record {RECORD_NAME} already deleted.")
#                 except Exception as e:
#                     print(f"Registry record delete: {e}")
#                 break
#         else:
#             print(f"Registry record {RECORD_NAME} not found (already deleted).")
#     except agentcore_control.exceptions.ResourceNotFoundException:
#         print("Registry already deleted; skipping record cleanup.")
#     except Exception as e:
#         print(f"List registry records: {e}")
#
# # ── Phase 2: Delete AgentCore Runtime (control plane API) and poll ──────────
# agent_runtime_id, agent_runtime_arn = None, None
# try:
#     paginator = agentcore_control.get_paginator("list_agent_runtimes")
#     for page in paginator.paginate():
#         for rt in page.get("agentRuntimes", []):
#             if rt.get("agentRuntimeName") == RUNTIME_NAME:
#                 agent_runtime_id  = rt.get("agentRuntimeId")
#                 agent_runtime_arn = rt.get("agentRuntimeArn")
#                 break
#         if agent_runtime_id:
#             break
# except Exception as e:
#     print(f"List agent runtimes: {e}")
#
# if agent_runtime_id:
#     try:
#         agentcore_control.delete_agent_runtime(agentRuntimeId=agent_runtime_id)
#         print(f"Initiated delete of agent runtime: {agent_runtime_id}")
#         deadline = time.time() + 300  # 5 min
#         while time.time() < deadline:
#             try:
#                 agentcore_control.get_agent_runtime(agentRuntimeId=agent_runtime_id)
#                 time.sleep(10)
#             except agentcore_control.exceptions.ResourceNotFoundException:
#                 print("Agent runtime fully deleted.")
#                 break
#         else:
#             print("Timed out waiting for runtime to delete; continuing.")
#     except agentcore_control.exceptions.ResourceNotFoundException:
#         print("Agent runtime already deleted.")
#     except Exception as e:
#         print(f"Agent runtime delete: {e}")
# else:
#     print(f"Agent runtime {RUNTIME_NAME} not found (already deleted).")
#
# # ── Phase 3: Delete CloudWatch log delivery components ──────────────────────
# # Order: deliveries → destinations → sources (CloudWatch enforces this order)
# sources_for_runtime = []
# if agent_runtime_arn:
#     try:
#         paginator = logs_client.get_paginator("describe_delivery_sources")
#         for page in paginator.paginate():
#             for s in page.get("deliverySources", []):
#                 if agent_runtime_arn in s.get("resourceArns", []):
#                     sources_for_runtime.append(s)
#     except Exception as e:
#         print(f"List delivery sources: {e}")
#
# source_names = {s["name"] for s in sources_for_runtime}
# dest_arns_to_delete = set()
#
# if source_names:
#     try:
#         paginator = logs_client.get_paginator("describe_deliveries")
#         for page in paginator.paginate():
#             for d in page.get("deliveries", []):
#                 if d.get("deliverySourceName") in source_names:
#                     if d.get("deliveryDestinationArn"):
#                         dest_arns_to_delete.add(d["deliveryDestinationArn"])
#                     try:
#                         logs_client.delete_delivery(id=d["id"])
#                         print(f"Deleted delivery: {d['id']}")
#                     except Exception as e:
#                         print(f"Delete delivery ({d['id']}): {e}")
#     except Exception as e:
#         print(f"List deliveries: {e}")
#
# for dest_arn in dest_arns_to_delete:
#     dest_name = dest_arn.split(":delivery-destination:", 1)[-1] if ":delivery-destination:" in dest_arn else None
#     if dest_name:
#         try:
#             logs_client.delete_delivery_destination(name=dest_name)
#             print(f"Deleted delivery destination: {dest_name}")
#         except Exception as e:
#             print(f"Delete delivery destination ({dest_name}): {e}")
#
# for s in sources_for_runtime:
#     try:
#         logs_client.delete_delivery_source(name=s["name"])
#         print(f"Deleted delivery source: {s['name']}")
#     except Exception as e:
#         print(f"Delete delivery source ({s['name']}): {e}")
#
# # ── Phase 4: Delete CloudWatch log groups ───────────────────────────────────
# log_groups = [f"/aws/lambda/{LAMBDA_NAME}"]
# if agent_runtime_id:
#     log_groups += [
#         f"/aws/vendedlogs/bedrock-agentcore/runtime/APPLICATION_LOGS/{agent_runtime_id}",
#         f"/aws/vendedlogs/bedrock-agentcore/runtime/USAGE_LOGS/{agent_runtime_id}",
#     ]
# for lg in log_groups:
#     try:
#         logs_client.delete_log_group(logGroupName=lg)
#         print(f"Deleted log group: {lg}")
#     except logs_client.exceptions.ResourceNotFoundException:
#         pass
#     except Exception as e:
#         print(f"Log group delete ({lg}): {e}")
#
# # ── Phase 5: Delete Gateway Target ──────────────────────────────────────────
# if GATEWAY_ID:
#     try:
#         targets = agentcore_control.list_gateway_targets(gatewayIdentifier=GATEWAY_ID)
#         for t in targets.get("items", targets.get("targets", [])):
#             if t.get("name") == TARGET_NAME:
#                 try:
#                     agentcore_control.delete_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=t["targetId"])
#                     print(f"Deleted gateway target: {TARGET_NAME}")
#                 except agentcore_control.exceptions.ResourceNotFoundException:
#                     print(f"Gateway target {TARGET_NAME} already deleted.")
#                 except Exception as e:
#                     print(f"Gateway target delete: {e}")
#                 break
#     except agentcore_control.exceptions.ResourceNotFoundException:
#         print("Gateway already deleted; target removed implicitly.")
#     except Exception as e:
#         print(f"List gateway targets: {e}")
#
# # ── Phase 6: Delete Lambda function ─────────────────────────────────────────
# try:
#     lambda_client.delete_function(FunctionName=LAMBDA_NAME)
#     print(f"Deleted Lambda: {LAMBDA_NAME}")
# except lambda_client.exceptions.ResourceNotFoundException:
#     print(f"Lambda {LAMBDA_NAME} already deleted.")
# except Exception as e:
#     print(f"Lambda delete: {e}")
#
# # ── Phase 7: Delete IAM roles (Lambda + Runtime execution) ──────────────────
# for role_name in [LAMBDA_ROLE_NAME, RUNTIME_ROLE_NAME]:
#     try:
#         for ap in iam.list_attached_role_policies(RoleName=role_name).get("AttachedPolicies", []):
#             iam.detach_role_policy(RoleName=role_name, PolicyArn=ap["PolicyArn"])
#         for p in iam.list_role_policies(RoleName=role_name).get("PolicyNames", []):
#             iam.delete_role_policy(RoleName=role_name, PolicyName=p)
#         iam.delete_role(RoleName=role_name)
#         print(f"Deleted IAM role: {role_name}")
#     except iam.exceptions.NoSuchEntityException:
#         print(f"IAM role {role_name} already deleted.")
#     except Exception as e:
#         print(f"IAM role cleanup ({role_name}): {e}")
#
# # ── Phase 8: Delete ECR repository (force=True removes container images) ────
# try:
#     ecr.delete_repository(repositoryName=ECR_REPO_NAME, force=True)
#     print(f"Deleted ECR repository: {ECR_REPO_NAME}")
# except ecr.exceptions.RepositoryNotFoundException:
#     print(f"ECR repository {ECR_REPO_NAME} already deleted.")
# except Exception as e:
#     print(f"ECR delete: {e}")
#
# # ── Phase 9: Delete CodeBuild project (per-agent build project) ─────────────
# try:
#     codebuild.delete_project(name=CODEBUILD_PROJECT)
#     print(f"Deleted CodeBuild project: {CODEBUILD_PROJECT}")
# except Exception as e:
#     msg = str(e)
#     if "ResourceNotFoundException" in msg or "does not exist" in msg.lower():
#         print(f"CodeBuild project {CODEBUILD_PROJECT} already deleted.")
#     else:
#         print(f"CodeBuild delete: {e}")
#
# # ── Phase 10: Delete SSM parameters ─────────────────────────────────────────
# for param in ["refund_lambda_arn", "refund_gateway_target_id"]:
#     try:
#         ssm.delete_parameter(Name=f"{SSM_PREFIX}/{param}")
#         print(f"Deleted SSM parameter: {SSM_PREFIX}/{param}")
#     except ssm.exceptions.ParameterNotFound:
#         pass
#     except Exception as e:
#         print(f"SSM delete ({param}): {e}")
#
# print("\n✅ All Refund Agent resources cleaned up.")